In [1]:
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.cosmology import Planck15
import astropy.cosmology as cosmo
import astropy.units as u
from scipy.optimize import minimize
from tqdm import tqdm
from astropy.io import fits
from scipy import integrate
H0 =  72.3
Om = 0.3
Ok = 0.0
w = -1.0
zs = np.linspace(0.001, 5, 100)
zs = zs[1:] # We don't want z = 0
my_cosmo = cosmo.wCDM(H0=H0, Om0=Om, Ode0=1.0-Om-Ok)
dls_true = my_cosmo.luminosity_distance(zs).to(u.Mpc).value
ys = zs/(1+zs)

In [2]:
def to_statefinders(theta):
    # Default values for E0, E1, E2 (assuming minimum length is 3)
    H0= theta[0]
    E1 = theta[1]

    # Always calculate q0 and j0
    q0 = E1 - 1
    theta_final = []
    theta_final.append(H0)
    theta_final.append(q0)
    length=len(theta)

    if length>=3:
      E2=theta[2]
      j0 = E1**2 - 2*E1 + E2 + 1
      theta_final.append(j0)


    # Add s0 if the length of theta is at least 4
    if len(theta) >= 4:
        E3 = theta[3]
        s0 = -E1**3 + 3*E1**2 - 4*E1*E2 - 3*E1 + E2 - E3 + 1
        theta_final.append(s0)

    # Add l0 if the length of theta is at least 5
    if len(theta) >= 5:
        E4 = theta[4]
        l0 = E1**4 - 4*E1**3 + 11*E1**2*E2 + 6*E1**2 - E1*E2 + 7*E1*E3 - 4*E1 + 4*E2**2 + 2*E2 + E3 + E4 + 1
        theta_final.append(l0)
    if len(theta)>=6:
        E5=theta[5]
        p0=-E1**5 + 5*E1**4 - 26*E1**3*E2 - 10*E1**3 - 18*E1**2*E2 - 32*E1**2*E3 + 10*E1**2 - 34*E1*E2**2 - 18*E1*E2 - 24*E1*E3 - 11*E1*E4 - 5*E1 - 11*E2**2 - 15*E2*E3 + 2*E2 - 4*E3 - 4*E4 - E5 + 1
        theta_final.append(p0)

    # Append E0 at the end


    return theta_final


to_statefinders([E0,E1,E2,E3,E4,E5])

NameError: name 'E0' is not defined

In [2]:
def chi_square(theta,func, z, data, order):
    diff = (func(z, *theta[:order+1], order=order) - data)
    return np.sum(diff*diff)

# def chi_square_with_penalty(theta, func, z, data, order, penalty=1e10):
#     # Original chi-square calculation
#     chi_sq = chi_square(theta, func, z, data, order)

#     penalty_term = 0
#     for i in range(len(theta)):
#         if lower_bounds[i] is not None and theta[i] < lower_bounds[i]:
#             penalty_term += penalty * (lower_bounds[i] - theta[i])**2
#         if upper_bounds[i] is not None and theta[i] > upper_bounds[i]:
#             penalty_term += penalty * (theta[i] - upper_bounds[i])**2

#     return chi_sq + penalty_term

def EIS(z,H0,E1,E2=None,E3=None,E4=None,E5=None,order=None):
    
    
    dL=np.full(z.shape,np.nan)
    E0=1

    def func(z,E1,E2,E3,E4,E5,order):
        if order ==1:
            y=1/(E0+E1*z)
        elif order==2:
            y=1/(E0+E1*z+(E2*z**2))
        elif order==3:
            y=1/(E0+E1*z+(E2*z**2)+(E3*z**3))
        elif order==4:
            y = 1 / (E0 + E1 * z + (E2 * z**2) + (E3 * z**3)+ (E4 * z**4))
        elif order==5:
            y = 1 / (E0 + E1 * z + (E2 * z**2) + (E3 * z**3)+ (E4 * z**4) + (E5 * z**5))
        return y


    for  i,z_val in  np.ndenumerate(z):
        dL[i]=(1+z_val)*integrate.quad(func,0,z_val,args=(E1,E2,E3,E4,E5,order))[0]
    
    return (c/H0)*dL




def to_cosmographic(H0, Om, Ok, w):
    q0 = 1.0/2.0 *(1.0-Ok-3.0*(-1.0+Om+Ok)*w)
    j0 = 1 - Ok + (9/2)*(1.0-Om-Ok)*w*(1+w)
    s0 = -7/2 + 4*Ok -Ok**2/2 + (-81/4 + 15*Ok/4)*(1-Ok-Om)*w + (9*Ok/4 -117/4 -27*(1-Ok-Om)/4)*(1-Ok-Om)*w**2 -(27/4)*(3-Ok -Om)*(1-Ok-Om)*w**3
    c0 = 35/2 -23*Ok +11*Ok**2/2 +(489/4 -189*Ok/4)*(1-Ok-Om)*w +(207 + 189*(1-Ok -Om)/2 -99*Ok/2)*(1-Ok-Om)*w**2 +(621/4 + 162*(1- Ok - Om) -81/4*Ok)*(1- Ok - Om)*w**3 +81/2 *(1+2*(1-Ok -Om))*(1-Ok-Om)*w**4
    p0 = 1/4*(-455 +681*Ok -237*Ok**2 +11*Ok**3) +(-7407/8 +2187*Ok/4 -255*Ok**2/8)*(1.0-Om-Ok)*w \
         +(-6849/4 + 1449*Ok/2 -99*Ok**2/4 -9315*(1.0-Om-Ok)/8 +945*Ok*(1.0-Om-Ok)/8)*(1.0-Om-Ok)*w**2 \
         +(-13041/8 +1971*Ok/4 -81*Ok**2/8 -5103*(1.0-Om-Ok)/2 +621*Ok*(1.0-Om-Ok)/4 -567*(1.0-Om-Ok)*(1.0-Om-Ok)/4)*(1.0-Om-Ok)*w**3 \
         +(-729 +243*Ok/2 -17577*(1.0-Om-Ok)/8 +567*Ok*(1.0-Om-Ok)/8 -243*(1.0-Om-Ok)*(1.0-Om-Ok))*(1.0-Om-Ok)*w**4 \
         -243/4*(2+11*(1.0-Om-Ok)+2*(1.0-Om-Ok)*(1.0-Om-Ok))*(1.0-Om-Ok)*w**5
    return H0, q0, j0, s0, c0, p0








In [344]:
# it is working!

n = 1              # number of runs
N = 2               # number of z_max we want
K = False
w = False
zmin = 0.0008
MET = 7
fun = chi_square

# Minimizer method selection
if MET==1:
    met='Nelder-Mead' #very slow
# elif MET==2:
#     met='SLSQP'
# elif MET==3:
#     met='Powell' # ultra slow
elif MET==4:
    met="trust-constr"
elif MET==5:
    met="TNC"
elif MET ==6:
    met="COBYLA"
elif MET ==7:
    met="L-BFGS-B" #fast
# elif MET==8:
#     met="Newton-CG"
# elif MET==9:
#     met='BFGS'
# elif MET==10:
#     met='CG'
# elif MET==11:
#     met='trust-exact'

# Define the range of zmax values
z_max_array = np.linspace(0.2, 2, N)
#order_list = [3,4,5,6]
order_list = [2,3,4,5]
# Initialize lists to store results
min_z = []
min_y = []
min_log = []
min_pade = []
true = []
success = []
chisquare = []
min_eis=[]



for ord, order in enumerate(order_list):

    min_eis_order = np.full((len(z_max_array), n, order+1), np.nan)
    true_order = np.full((len(z_max_array), n, order+1), np.nan)
    true_start = np.full((len(z_max_array), n, order+1), np.nan)

    success_order = np.full((len(z_max_array), n, order+1), np.nan)

    # Loop over z_max_array
    for idx, zmax in enumerate(tqdm(z_max_array)):
        for i in range(n):

            H0_true = np.random.uniform(65, 75)
            Om_true = np.random.uniform(0.2, 0.4)
            if K:
                Ok_true = np.random.uniform(-0.05, 0.05)
            else:
                Ok_true = 0
            if w:
                w_true = np.random.uniform(-1.5,-0.5)
            else:
                w_true = -1
            pars = to_cosmographic(H0_true, Om_true, Ok_true, w_true)
            q0,j0,s0,c0,p0=pars[1:]
            E0=1
            E1=q0 + 1
            OK=Ok_true
            E2=(-OK/2 + j0/2 - q0**2/2)*2
            E3=(-2*j0*q0/3 - j0/2 + q0**3/2 + q0**2/2 - s0/6)*6
            E4=(21*OK**2/4 + 99*OK*j0/2 - 223*OK*q0**2 - 793*OK*q0/4 - 99*OK/4 + c0/24 - j0**2/6 + 25*j0*q0**2/24 + 4*j0*q0/3 + j0/2 - 5*q0**4/8 - q0**3 - q0**2/2 + 7*q0*s0/24 + s0/3)*24
            E5=(399*OK**2*q0/4 + 231*OK**2/4 - 6953*OK*j0*q0/6 - 769*OK*j0 + 2070*OK*q0**3 + 6039*OK*q0**2/2 + 2639*OK*q0/2 - 539*OK*s0/6 + 70*OK - 11*c0*q0/120 - c0/8 + 7*j0**2*q0/12 + j0**2/2 - 7*j0*q0**3/4 - 25*j0*q0**2/8 - 2*j0*q0 + j0*s0/8 - j0/2 - p0/120 + 7*q0**5/8 + 15*q0**4/8 + 3*q0**3/2 - q0**2*s0/2 + q0**2/2 - 7*q0*s0/8 - s0/2)*120

            pars_new=[H0_true,E1,E2,E3,E4,E5]
            pars_cosmo=[H0_true,q0,j0,s0,c0,p0]
            
            
            for j in range(order+1):
                true_order[idx, i, j] = pars_cosmo[j]
                true_start[idx, i, j] = pars_new[j]

            zs = np.arange(zmin, zmax, 0.005)
            my_cosmo = cosmo.wCDM(H0=H0_true, Om0=Om_true, Ode0=1.0 - Om_true - Ok_true, w0=w_true)
            dls_true = my_cosmo.luminosity_distance(zs).to(u.Mpc).value

            # Minimizer parameters
            start = true_start[idx, i] + abs(true_start[idx, i]) * np.random.normal(0, 0.05, order+1)


            result_eis = minimize(fun, start, args=(EIS, zs, dls_true, order), method=met, options={'maxiter': 200000}, tol=1e-12)
            success_order=result_eis.success
            if result_eis.success == True:

                min_eis_order[idx, i, :] = to_statefinders(result_eis.x)




    # Append current order results to lists
    min_eis.append(min_eis_order)
    true.append(true_order)
    success.append(success_order)
    #chisquare.append(chisq_order)

 50%|██████████████████████████████████████████                                          | 1/2 [00:00<00:00,  2.45it/s]C:\Users\Animesh\AppData\Local\Temp\ipykernel_9508\380494875.py:39: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  dL[i]=(1+z_val)*integrate.quad(func,0,z_val,args=(E1,E2,E3,E4,E5,order))[0]
C:\Users\Animesh\AppData\Local\Temp\ipykernel_9508\380494875.py:39: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  dL[i]=(1+z_val)*integrate.quad(func,0,z_val,args=(E1,E2,E3,E4,E5,order))[0]
C:\Users\Animesh\AppData\Local\Temp\ipykernel_9508\380494875.py:39: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order 

# Generate files

In [346]:
min_eis

[array([[[72.3620482 , -0.50054422,  0.98177076]],
 
        [[66.190482  , -0.4687283 ,  0.69849054]]]),
 array([[[68.92013549, -0.40330006,  0.99548985, -0.77201823]],
 
        [[73.85033913, -0.6224909 ,  1.03117368,  0.13659689]]]),
 array([[[67.54892787, -0.57727431,  1.00001629, -0.27920678,
           2.87337955]],
 
        [[68.33505974, -0.69718117,  1.08106639,  0.43021179,
           2.61691154]]]),
 array([[[ 74.08359971,  -0.53784098,   1.00384031,  -0.3461284 ,
            3.13730797, -10.86382736]],
 
        [[ 70.6176952 ,  -0.44634847,   0.73176477,  -1.27963621,
            4.54145513, -19.0806066 ]]])]

In [348]:
true

[array([[[72.31288715, -0.50127836,  1.        ]],
 
        [[66.88814138, -0.55470822,  1.        ]]]),
 array([[[68.87273462, -0.40352456,  1.        , -0.78942632]],
 
        [[73.8186573 , -0.6224605 ,  1.        , -0.1326185 ]]]),
 array([[[ 6.75021649e+01, -5.77256609e-01,  1.00000000e+00,
          -2.68230174e-01,  2.91775863e+00]],
 
        [[ 6.82511188e+01, -6.88532013e-01,  1.00000000e+00,
           6.55960404e-02,  2.20500981e+00]]]),
 array([[[ 74.03218208,  -0.53770357,   1.        ,  -0.38688928,
            3.20690077, -11.4473217 ]],
 
        [[ 70.73329249,  -0.48170348,   1.        ,  -0.55488955,
            3.6483807 , -14.1707786 ]]])]

In [350]:
success

[True, True, True, True]

In [ ]:
import awkward as ak

min_z_awkward = ak.Array(min_z)
min_y_awkward = ak.Array(min_y)
min_log_awkward = ak.Array(min_log)
min_pade_awkward = ak.Array(min_pade)
true_awkward = ak.Array(true)
success_awkward = ak.Array(success)
chisquare_awkward = ak.Array(chisquare)


# In[13]:


stri='run1_MET4_simseq50_zmax2_10'
ak.to_parquet(min_z_awkward, f"min_z_awkward_{stri}.parquet")
ak.to_parquet(min_y_awkward, f"min_y_awkward_{stri}.parquet")
ak.to_parquet(min_log_awkward, f'min_log_awkward_{stri}.parquet')
ak.to_parquet(min_pade_awkward, f'min_pade_awkward_{stri}.parquet')
ak.to_parquet(true_awkward, f'true_awkward_{stri}.parquet')
ak.to_parquet(success_awkward, f'success_awkward_{stri}.parquet')
ak.to_parquet(chisquare_awkward, f'chisquare_awkward_{stri}.parquet')